In [ ]:
import GEOparse

gse = GEOparse.get_GEO(geo="GSE10245", destdir="./")

In [ ]:
gse.metadata

Non-small cell lung cancer (NSCLC) can be classified into the major subtypes adenocarcinoma (AC) and squamous cell carcinoma (SCC) subtypes. Although explicit molecular, histological and clinical characteristics have been reported for both subtypes, no specific therapy exists so far. However, the characterization of suitable molecular targets holds great promises to develop novel therapies in NSCLC. In the present study, global gene expression profiling of 58 human high grade NSCLC specimens revealed large transcriptomic differences between AC and SCC subtypes: More than 1.700 genes were found to be differentially expressed.
 

In [ ]:
for gene in gse.gpls.items():
    print(gene)


In [ ]:
import numpy as np
data = gse.pivot_samples('VALUE')

adeno_samples = []
squamous_samples = []

for gsm_id, gsm in gse.gsms.items():
    status = gsm.metadata['characteristics_ch1'][0]
    if 'adenocarcinoma' in status:
        adeno_samples.append(gsm_id)
    elif 'squamous cell carcinoma' in status:
        squamous_samples.append(gsm_id)



In [ ]:
adeno_mean = data[adeno_samples].mean(axis=1)
squamous_mean = data[squamous_samples].mean(axis=1)
diff = adeno_mean - squamous_mean

In [ ]:
results = pd.DataFrame({
    'adeno_avg': adeno_mean,
    'squamous_avg': squamous_mean,
    'diff': diff
})

top_up = results.sort_values('diff', ascending=False).head(10)
top_down = results.sort_values('diff', ascending=True).head(10)

In [ ]:
top_down

In [ ]:
top_up

Choosen genes:
244056_at
201820_at

In [ ]:
genes_to_plot = ["244056_at", "201820_at"]
plot_df = data.loc[genes_to_plot].T.reset_index()


In [ ]:
plot_df['status'] = np.select([plot_df['name'].isin(adeno_samples),plot_df['name'].isin(squamous_samples)], ['A','S'], default='None')
plot_df

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.boxplot(data=plot_df, x='status', y='244056_at')
plt.title("Gene Expression Comparison: Adeno vs Squamous")
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.boxplot(data=plot_df, x='status', y='201820_at')
plt.title("Gene Expression Comparison: Adeno vs Squamous")
plt.show()

In [ ]:
# 244056_at Gene Symbol: SFTA2

gpl = gse.gpls["GPL570"].table
gpl[gpl['ID'] == "244056_at"]

In [ ]:
# 201820_at Gene Symbol: KRT5
gpl[gpl['ID'] == "201820_at"]

2. Sequence Extraction

In [8]:
import biotite.database.entrez as entrez
import biotite.sequence.io.genbank as gb
from biotite.sequence import NucleotideSequence, ProteinSequence
from typing import Optional

def fetch_gene_cds(
    gene_name: str,
    organism: str = "Homo sapiens",
    number: int = 1,
) -> Optional[str]:
    """
    Fetch a nucleotide GenBank record from NCBI Entrez using Biotite and return
    concatenated CDS nucleotide sequence(s) from the first hit.
    """

    # Build Entrez query
    query = (
        entrez.SimpleQuery(gene_name, "Gene Name")
        & entrez.SimpleQuery(organism, "Organism")
    )

    # Search UIDs
    uids = entrez.search(query, db_name="Nucleotide", number=number)

    if not uids:
        print("Gene not found.")
        return None

    uid = uids[0]

    # Fetch GenBank
    gb_io = entrez.fetch_single_file([uid], None, "Nucleotide", "gb", ret_mode="text")

    # Parse GenBank
    gb_file = gb.GenBankFile.read(gb_io)
    seq = gb.get_sequence(gb_file)
    annotation = gb.get_annotation(gb_file)
    print(annotation)
    # Extract CDS fragments
    cds_fragments = []
    for feature in annotation:
        if feature.key != "CDS":
            continue

        for loc in feature.locs:
            start = loc.first - 1
            stop = loc.last
            frag = seq[start:stop]
            cds_fragments.append(str(frag))

    if not cds_fragments:
        print("No CDS/exons found for this gene.")
        return None

    print(cds_fragments)
    return "".join(cds_fragments)

# Obtain mRNA sequence for BRCA1 gene and translate to protein sequence

sfta = NucleotideSequence(fetch_gene_cds("SFTA2"))
krt = NucleotideSequence(fetch_gene_cds("KRT5"))


Annotation([Feature("gene", [Location(2651502, 2652853, strand=Location.Strand.REVERSE, defect=Location.Defect.BEYOND_LEFT|BEYOND_RIGHT)], qual={'gene': 'POU5F1', 'locus_tag': 'LCF51_0116'}), Feature("CDS", [Location(3200549, 3200772, strand=Location.Strand.FORWARD, defect=Location.Defect.NONE), Location(3198470, 3198592, strand=Location.Strand.FORWARD, defect=Location.Defect.NONE), Location(3198324, 3198378, strand=Location.Strand.FORWARD, defect=Location.Defect.NONE)], qual={'gene': 'LY6G6D', 'locus_tag': 'LCF51_0164', 'codon_start': '1', 'product': 'LY6G6D', 'protein_id': 'UQL51159.1', 'translation': 'MKPQFVGILLSSLLGAALGNRMRCYNCGGSPSSSCKEAVTTCGE GRPQPGLEQIKLPGNPPVTLIHQHPACVAAHHCNQVETESVGDVTYPAHRDCYLGDLC NSAVASHVAPAGILAAAATALTCLLPGLWSG'}), Feature("mRNA", [Location(3444504, 3444694, strand=Location.Strand.FORWARD, defect=Location.Defect.NONE), Location(3446721, 3446867, strand=Location.Strand.FORWARD, defect=Location.Defect.NONE), Location(3443528, 3443645, strand=Location.Strand.FOR

3. Translation to protein

In [9]:
from biotite.sequence import CodonTable
codon_table = {
    'A': ['GCT','GCC','GCA','GCG'],
    'C': ['TGT','TGC'],
    'D': ['GAT','GAC'],
    'E': ['GAA','GAG'],
    'F': ['TTT','TTC'],
    'G': ['GGT','GGC','GGA','GGG'],
    'H': ['CAT','CAC'],
    'I': ['ATT','ATC','ATA'],
    'K': ['AAA','AAG'],
    'L': ['TTA','TTG','CTT','CTC','CTA','CTG'],
    'M': ['ATG'],
    'N': ['AAT','AAC'],
    'P': ['CCT','CCC','CCA','CCG'],
    'Q': ['CAA','CAG'],
    'R': ['CGT','CGC','CGA','CGG','AGA','AGG'],
    'S': ['TCT','TCC','TCA','TCG','AGT','AGC'],
    'T': ['ACT','ACC','ACA','ACG'],
    'V': ['GTT','GTC','GTA','GTG'],
    'W': ['TGG'],
    'Y': ['TAT','TAC'],
    '*': ['TAA','TAG','TGA']
}

reverse_codon_table = {i: k for k, v in codon_table.items() for i in v}
new_codon_table = CodonTable(
    codon_dict = reverse_codon_table,
    starts=['ATG']
)

new_codon_table

sfta_seq = sfta.translate(codon_table=new_codon_table, met_start=True)
krt_seq = krt.translate(codon_table=new_codon_table, met_start=True)
